In [1]:
import xml.etree.ElementTree as ET

import json
import re
from pathlib import Path
from enum import Enum

from dataclasses import dataclass
from typing import Union, List, Optional,Tuple
from re import Match
from ast import pattern


import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import sys

import requests



## CiViC extraction

In [23]:
sys.path.insert(0, str(Path(".").resolve().parent))
import src.preprocessing.utils.biomarker_CiViC_filter as biomarker_CiViC_filter
import src.preprocessing.utils.parsing as parsing_ClinicalTrials
import src.preprocessing.utils.prompt_call as prompt_call_ClinicalTrials
ClinicalTrial=parsing_ClinicalTrials.ClinicalTrial

In [ ]:
path="../data/CiViC/nightly-VariantSummaries.tsv"
# expanded is just to have all those possible alliases, and our mod is the cleaned version of the variant
civic, civic_df, expanded_df,synonyms = biomarker_CiViC_filter.main_(path, type_analysis="CiViC")


In [ ]:
#need to implement this on filtering

## Parsing clinical trials

In [25]:
clinical_trials_path="../data/synthetic/trials/TREC_2023/ClinicalTrials.2023-05-08.trials5/NCT0585xxxx/"
trials = parsing_ClinicalTrials.process_clinical_trials(clinical_trials_path, n=15, word_filter=["cancer", "tumor", "neoplasm"])


## CT filtering based on biomarkers and token distribution 


 Token distributions in real-world clinical trial data are heavily right-skewed — most trials have a moderate length, but a few have extremely long, verbose eligibility criteria sections. These outliers can cause problems like exceeding model context windows, inflating loss during fine-tuning, or introducing instability in training.
 
What "uniformity" means here: They're not requiring all trials to be identical in length, but rather removing trials that are so far out in the tail (e.g., 3+ standard deviations above the mean, or above some hard token cap) that they'd disrupt the fine-tuning process — either technically (context length limits) or statistically (dominating gradient updates).

## Prompts generation

In [26]:
import json, sys, os

from tqdm import tqdm
import ast
import logging
import openai
import torch
from transformers import AutoTokenizer

print("CUDA available:", torch.cuda.is_available())

from vllm import LLM, SamplingParams
import torch.multiprocessing as mp
mp.set_start_method('spawn', force=True)

logging.basicConfig(level=logging.INFO)
logging.getLogger("vllm").setLevel(logging.WARNING)



CUDA available: True


In [9]:
dnf_prompt_dir = Path("../src/prompts/trialQuestionsPrompt.py")
from transformers import AutoConfig

PROMPT_DNF_TEMPLATE = prompt_call_ClinicalTrials.load_DNF_prompt(dnf_prompt_dir)
model = "Qwen/Qwen1.5-14B-Chat"

config= {
          	'model_name': model,
          	'temperature': 0.0,
               'max_tokens': 1024,
               'max_context': 0
          }
def_conf=AutoConfig.from_pretrained(config['model_name'])
config['max_context'] = getattr(def_conf, "max_position_embeddings", None)


In [11]:
# Example usage:
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  
llm = LLM(
     model=config['model_name'],
     gpu_memory_utilization=0.88,
     dtype='bfloat16',
     max_model_len=13472
)


(EngineCore pid=3646452) INFO 07-06 17:31:40 [core.py:113] Initializing a V1 LLM engine (v0.23.0) with config: model='Qwen/Qwen1.5-14B-Chat', speculative_config=None, tokenizer='Qwen/Qwen1.5-14B-Chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=13472, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:01<00:11,  1.61s/it]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:03<00:10,  1.69s/it]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:05<00:08,  1.73s/it]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:06<00:06,  1.73s/it]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:08<00:05,  1.74s/it]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:10<00:03,  1.73s/it]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:11<00:01,  1.61s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:12<00:00,  1.33s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:12<00:00,  1.55s/it]
(EngineCore pid=3646452) 


(EngineCore pid=3646452) INFO 07-06 17:32:22 [default_loader.py:397] Loading weights took 12.46 seconds
(EngineCore pid=3646452) INFO 07-06 17:32:22 [gpu_model_runner.py:5187] Model loading took 26.43 GiB memory and 39.402680 seconds
(EngineCore pid=3646452) INFO 07-06 17:32:28 [backends.py:1089] Using cache directory: /users/jcc2340/.cache/vllm/torch_compile_cache/58602afa2a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3646452) INFO 07-06 17:32:28 [backends.py:1148] Dynamo bytecode transform time: 5.53 s
(EngineCore pid=3646452) INFO 07-06 17:32:30 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=3646452) INFO 07-06 17:32:35 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 5.79 s
(EngineCore pid=3646452) INFO 07-06 17:32:38 [decorators.py:708] saved AOT compiled function to /users/jcc2340/.cache/vllm/torch_compile_cache/torch_aot_compile/235ba7764416c2efa1f16c94d5d5001b38c424a1f970351bd702c803e1a80eea/rank_0_

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.60it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.66it/s]


(EngineCore pid=3646452) INFO 07-06 17:32:53 [gpu_model_runner.py:6585] Graph capturing finished in 7 secs, took 0.54 GiB
(EngineCore pid=3646452) INFO 07-06 17:32:53 [gpu_worker.py:639] CUDA graph pool memory: 0.54 GiB (actual), 0.62 GiB (estimated), difference: 0.08 GiB (15.1%).
(EngineCore pid=3646452) INFO 07-06 17:32:53 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=3646452) INFO 07-06 17:32:54 [core.py:306] init engine (profile, create kv cache, warmup model) took 31.46 s (compilation: 14.94 s)
(EngineCore pid=3646452) INFO 07-06 17:32:56 [vllm.py:999] Asynchronous scheduling is enabled.
(EngineCore pid=3646452) INFO 07-06 17:32:56 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


In [27]:
for trial in tqdm(trials, desc="Generating DNF", unit="trial", dynamic_ncols=True):
    prompt_call_ClinicalTrials.generate_DNF(trial, PROMPT_DNF_TEMPLATE, config, type_run='hpc', llm=llm)


Generating DNF:   0%|          | 0/4 [00:00<?, ?trial/s]

<|im_start|>system
You are an expert in clinical trial eligibility criteria. Output only valid JSON, no extra text or explanations.<|im_end|>
<|im_start|>user
You are an expert clinical trial data extractor. Convert the given INCLUSION and EXCLUSION CRITERIA into a set of clear, grammatically correct YES/NO questions, then build a logical (DNF) expression over those questions to determine trial eligibility.

INPUT FORMAT:
Two strings — "Inclusion Criteria" and "Exclusion Criteria" (the latter's header starts with "!"). Each is formatted as "Header: criteria1 criteria2 ...", with individual criteria separated by newlines, bullets, hyphens, or numbers.

QUESTION RULES:
- One question per single criterion. If a line contains multiple criteria, split it into separate questions (e.g. "18-65 years old and ECOG 0-1" > "Is the patient between 18 and 65 years old?" + "Does the patient have an ECOG performance status of 0 or 1?").
- Questions must be answerable independently, and phrased only in

Generating DNF:  25%|██▌       | 1/4 [00:20<01:01, 20.51s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Is the patient histologically confirmed with adenocarcinoma of the prostate?",', '        "Q2": "Is the patient\'s prostate cancer unfavorable intermediate-risk or high-risk, with the specified definitions?",', '        "Q3": "Is the patient\'s ECOG performance status between 0 and 2?",', '        "Q4": "Is the patient at least 18 years old?",', '        "Q5": "Has the patient undergone a 3 Tesla prostate MRI within the last 12 months?",', '        "Q6": "Does the patient have planned androgen deprivation therapy meeting the specified criteria?",', '        "Q7": "Is there evidence of pelvic nodal or distant metastases?",', '        "Q8": "Is there discordance between pre-enrollment MRI and biopsy findings with Gleason 4+3 (grade group ≥ 3)?",', '        "Q9": "Has the patient started androgen deprivation therapy more than 60 days prior to enrollment?",', '        "Q10": "Does the patient intend to electively treat pelvic lymph nod

Generating DNF:  50%|█████     | 2/4 [00:43<00:43, 21.86s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Is the patient over 18 years old?",', '        "Q2": "Does the patient have evidence of evaluable disease with a contrast enhancing or non-enhancing lesion > 1 cubic centimetre?",', '        "Q3": "Is the patient in Cohort 1 (IDH mutant glioma, regardless of prior treatment)?",', '        "Q4": "Is the patient in Cohort 2 (recurrent IDH mutant glioma before surgical resection)?",', '        "Q5": "Are prior MR scans available for tumor assessment?",', '        "Q6": "Does the patient have a life expectancy of more than 8 weeks?",', '        "Q7": "Is the patient\'s Karnofsky performance status greater than 70?",', '        "Q8": "Does the patient have adequate renal function with a creatinine level < 1.5 mg/dL and the test performed within 60 days?",', '        "Q9": "Does the patient have any significant medical illness that cannot be adequately controlled?",', '        "Q10": "Does the patient have NYHA Grade II or greater conges

Generating DNF:  75%|███████▌  | 3/4 [00:54<00:16, 16.84s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Does the participant provide written informed consent?",', '        "Q2": "Is the participant at least 70 years old?",', '        "Q3": "Is the participant male or female?",', '        "Q4": "Does the patient have a confirmed diagnosis of colorectal cancer?",', '        "Q5": "Is the patient awaiting major surgery?",', '        "Q6": "Does the patient have a clinical need for emergency intervention?",', '        "Q7": "Does the patient have a MMSE score of 20 or higher?",', '        "Q8": "Does the patient have an ADL score of 3 or higher?",', '        "Q9": "Is the patient\'s colorectal neoplasia stage IV?"', '    },', '    "DNF_LOGICAL_EXPRESSION": "not Q6 and not Q7 and not Q8 and Q1 and Q2 and Q3 and Q4 and Q5",', '    "DNF_LOGICAL_EXPRESSION_REASONING": "Patient is eligible if they meet all inclusion criteria (Q1-5) and none of the exclusion criteria (Q6-9, negated).",', '    "INCLUSION_BIOMARKER": "None",', '    "EXCLUSION_BI

Generating DNF: 100%|██████████| 4/4 [01:05<00:00, 16.25s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Is the patient between 18 and 75 years old?",', '        "Q2": "Does the patient have a pathologically-confirmed diagnosis of HR+ and HER2- breast cancer?",', '        "Q3": "Has the patient received any previous systematic antitumor therapy for locally advanced or metastatic breast cancer?",', '        "Q4": "Does the patient have an ECOG performance status of 0 or 1?",', '        "Q5": "Do the patient\'s organ and marrow functions meet the criteria?",', '        "Q6": "Does the patient agree to sign the informed consent?"', '    },', '    "DNF_LOGICAL_EXPRESSION": "Q1 and Q2 and not Q3 and Q4 and Q5 and Q6",', '    "DNF_LOGICAL_EXPRESSION_REASONING": "The patient must meet all inclusion criteria and none of the exclusion criteria.",', '    "INCLUSION_BIOMARKER": "None",', '    "EXCLUSION_BIOMARKER": "CDK4/6 inhibitor, GB491 allergy, HER2 positive disease, central nervous system metastases, major surgery within 4 weeks, previous c

# Embeddings and vector stores


In [16]:
import chromadb
from chromadb import Collection
import pickle
from transformers import AutoTokenizer, AutoModel
tokenizer_med=AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
model_med=AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1920.78it/s]


In [ ]:
def embed_text(text: str,tokenizer=tokenizer_med, model=model_med) -> np.ndarray:
    '''Embed a string of text [chunk]
    :param text: str - the text to be embedded
	:param tokenizer: the tokenizer to use for embedding, defaults to the MedCPT tokenizer
	:param model: the model to use for embedding, defaults to the MedCPT model
    :return: numpy array representing the embedding of the input text float32
    '''
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", 
                           truncation=True, padding=True,
                           max_length=512)
        embeddings = model(**inputs).last_hidden_state[:,0,:].squeeze().numpy() # we take the embedding of the [CLS] token, which is the first token in the input sequence
    return embeddings.astype('float32')

def _parse_questions_from_json(trial:ClinicalTrial) -> List[str]:
	"""Parse questions from the JSON representation of a trial.
	:param trial: ClinicalTrial - the trial object containing the JSON representation
	:return: List[str] - a list of questions extracted from the trial's JSON representation
	"""
	questions = []
	if trial.dnf_representation:
		for q in trial.dnf_representation:
			if q and re.match(r'^\s+"Q\d+"', q):
				string_q = re.split(r'"Q\d+":', q.strip())[1].strip()
				questions.append(string_q)
	return questions

def _add_questions(trial:ClinicalTrial,  tokenizer: AutoTokenizer, model: AutoModel,collection: Collection) -> None:
	'''Embed questions from a trial and add them to the specified ChromaDB collection.
	:param trial: ClinicalTrial - the trial object containing the questions to be embedded 
	:param tokenizer: AutoTokenizer - the tokenizer to use for embedding :param model: AutoModel - the model to use for embedding
	:param collection: Collection - the ChromaDB collection to which the embedded questions will be added
	:return: None'''

	threshold = 0.96
	embeddings,documents,metadata,ids = [],[],[],[]
	questions = _parse_questions_from_json(trial)
	for i, q in tqdm(enumerate(questions), total=len(questions), desc="Processing questions", unit="question", dynamic_ncols=True):
		e = embed_text(q, tokenizer_med, model_med)
		#print(f"Embedding for question {i}: {e}")
		if len(embeddings) > 0 : 
			existing_embeddings = np.array(embeddings)
			
			norm_new =e / np.linalg.norm(e)
			norm_existing = existing_embeddings/ np.linalg.norm(existing_embeddings, axis=1, keepdims=True)
			cosine_similarities = norm_existing @ norm_new
			if np.any(cosine_similarities > threshold):
				continue  # Skip adding this question if it's too similar to existing ones
		embeddings.append(e.tolist())
		documents.append(q)
		metadata.append({
			"trial_id": trial.name_id
			})
		ids.append(f"{trial.name_id}_Q{i}")
	if embeddings:
		print(f"Adding {len(embeddings)} questions to ChromaDB collection.")
		collection.add(ids=ids, documents=documents, metadatas=metadata, embeddings=embeddings)

def generate_chromaDB_CT(processed_trials: List[ClinicalTrial], 
				  tokenizer:AutoTokenizer, 
				  model: AutoModel,
				  save_dir:str) -> chromadb.Client:
	'''Generate a ChromaDB collection from a list of ClinicalTrial objects
	:param processed_trials: List[ClinicalTrial] - the list of ClinicalTrial objects to be added to the ChromaDB collection
	:param tokenizer: AutoTokenizer - the tokenizer to use for embedding, defaults to the MedCPT tokenizer
	:param model: AutoModel - the model to use for embedding, defaults to the MedCPT model
	:param save_dir: str - the directory where the ChromaDB collection will be saved
	'''
	from datetime import datetime
	timestamp = datetime.now().strftime("%Y%m%d")

	# Make directory if it doesn't exist
	save_path = Path(save_dir)
	if not save_path.exists():
		save_path.mkdir(parents=True, exist_ok=True)
	
	# Save the processed trials to a pickle file
	if (save_path / f"processed_trials_{timestamp}.pkl").exists():
		print(f"Warning: processed_trials_{timestamp}.pkl already exists and will be overwritten.")
	else:
		with open(save_path / f"processed_trials_{timestamp}.pkl", "wb") as f:
			pickle.dump(processed_trials, f) # In here we already have the trial information, we jsut need to add the embeddings and metadata to the ChromaDB collection

	# Create a new ChromaDB client and collection
	client = chromadb.PersistentClient(path=str(save_path / "chromaDB_trials"))
	collection = client.get_or_create_collection("clinical_trials")


	# Iterate over each trial and add it to the collection
	for trial in tqdm(processed_trials, desc="Adding trials to ChromaDB", unit="trial", dynamic_ncols=True):
		# Embed the DNF representation of the trial
		try:
			_add_questions(trial, tokenizer, model, collection)
		except Exception as e:
			print(f"Error occurred while processing trial {trial.name_id}: {e}")
	return client


| Field     | Value                                |
|-----------|---------------------------------------|
| id        | NCT05851235.xml_Q0                    |
| document  | "Is the patient over 18 years old?"   |
| metadata  | {"trial_id": "NCT05851235.xml"}       |
| embedding | MedCPT vector (768-dim)               |

In [72]:
generate_chromaDB_CT(trials, tokenizer_med, model_med, save_dir=Path("../database/chromadb/"))

Adding trials to ChromaDB:  25%|██▌       | 1/4 [00:23<01:09, 23.29s/trial]

Adding 16 questions to ChromaDB collection.


Adding trials to ChromaDB:  50%|█████     | 2/4 [00:48<00:48, 24.14s/trial]

Adding 17 questions to ChromaDB collection.


Adding trials to ChromaDB:  75%|███████▌  | 3/4 [01:01<00:19, 19.10s/trial]

Adding 9 questions to ChromaDB collection.


Adding trials to ChromaDB: 100%|██████████| 4/4 [01:09<00:00, 17.47s/trial]

Adding 6 questions to ChromaDB collection.
